
# PROYECTO PAMD – TERCER CORTE

## Auditoría y Monitoreo Continuo de la Contratación Estatal en Colombia

**Escuela Colombiana de Ingeniería Julio Garavito**

**Asignatura:** Procesamiento para el Análisis Masivo de Datos (PAMD)

### Objetivo general

Construir un flujo de procesamiento distribuido para analizar información de contratación pública de SECOP II, simular la llegada continua de contratos, garantizar la integridad de los resultados y presentar los principales hallazgos mediante un dashboard interactivo.

### Desarrollo

1. Articulación del flujo integral de contratación.
2. Simulación de flujo continuo y agrupación por ventanas temporales.
3. Resiliencia y resultados idempotentes.
4. Dashboard interactivo y visual del análisis realizado.


In [1]:

# Configuración inicial de Spark para el proyecto PAMD

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("PAMD_Tercer_Corte")
    .config(
        "spark.jars",
        "/home/jovyan/work/jars/hadoop-aws-3.3.4.jar,"
        "/home/jovyan/work/jars/aws-java-sdk-bundle-1.11.1026.jar"
    )
    .getOrCreate()
)

print("Spark iniciado correctamente")
print("Versión de Spark:", spark.version)


Spark iniciado correctamente
Versión de Spark: 3.5.0


In [2]:

# Configuración de conexión entre Spark y MinIO

hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.access.key", "minioadmin")
hadoop_conf.set("fs.s3a.secret.key", "minioadmin")
hadoop_conf.set("fs.s3a.path.style.access", "true")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")

print("Configuración de MinIO aplicada correctamente")


Configuración de MinIO aplicada correctamente


In [3]:
# Verificar que los cuatro indicadores siguen guardados en MinIO

ruta_indicadores = "s3a://raw/procesado/tercer_corte/indicadores/"

indicadores = [
    "departamentos",
    "entidades",
    "contratistas",
    "evolucion_anual"
]

for nombre in indicadores:
    df_prueba = spark.read.parquet(
        ruta_indicadores + nombre + "/"
    )

    print("\nIndicador:", nombre)
    print("Columnas:", df_prueba.columns)
    df_prueba.show(2, truncate=False)

print("\nVERIFICACIÓN DE INDICADORES FINALIZADA")


Indicador: departamentos
Columnas: ['departamento', 'cantidad_contratos', 'valor_total']
+--------------------------+------------------+---------------------+
|departamento              |cantidad_contratos|valor_total          |
+--------------------------+------------------+---------------------+
|Distrito Capital de Bogotá|596850            |1.3916642746257908E14|
|Antioquia                 |168153            |3.485657652990737E13 |
+--------------------------+------------------+---------------------+
only showing top 2 rows


Indicador: entidades
Columnas: ['entidad', 'cantidad_contratos', 'valor_total']
+----------------------------------------------------------------+------------------+--------------------+
|entidad                                                         |cantidad_contratos|valor_total         |
+----------------------------------------------------------------+------------------+--------------------+
|INVIAS                                                        

In [4]:

# Verificar el acceso al almacenamiento SECOP II en MinIO

ruta_secop = "s3a://raw/secop/"

# Obtener el sistema de archivos correspondiente a la ruta S3A
ruta = spark.sparkContext._jvm.org.apache.hadoop.fs.Path(ruta_secop)

sistema_archivos = ruta.getFileSystem(
    spark.sparkContext._jsc.hadoopConfiguration()
)

archivos = sistema_archivos.listStatus(ruta)

print("Archivos encontrados en MinIO:")

for archivo in archivos:
    print(archivo.getPath().toString())


Archivos encontrados en MinIO:
s3a://raw/secop/datos_secop.csv


In [5]:

# PUNTO 1. ARTICULACIÓN DEL FLUJO INTEGRAL DE CONTRATACIÓN

# Ruta de los datos procesados en el segundo corte
ruta_parquet = "s3a://raw/procesado/secop_parquet/"

# Cargar la información desde MinIO mediante Spark
df = spark.read.parquet(ruta_parquet)

# Consultar la estructura y el tamaño de la base
print("Base SECOP II cargada correctamente")
print("Cantidad de registros:", df.count())
print("Cantidad de columnas:", len(df.columns))

print("\nEstructura de los datos:")
df.printSchema()


Base SECOP II cargada correctamente
Cantidad de registros: 1777673
Cantidad de columnas: 85

Estructura de los datos:
root
 |-- Nombre Entidad: string (nullable = true)
 |-- Nit Entidad: string (nullable = true)
 |-- Departamento: string (nullable = true)
 |-- Ciudad: string (nullable = true)
 |-- Localización: string (nullable = true)
 |-- Orden: string (nullable = true)
 |-- Sector: string (nullable = true)
 |-- Rama: string (nullable = true)
 |-- Entidad Centralizada: string (nullable = true)
 |-- Proceso de Compra: string (nullable = true)
 |-- ID Contrato: string (nullable = true)
 |-- Referencia del Contrato: string (nullable = true)
 |-- Estado Contrato: string (nullable = true)
 |-- Codigo de Categoria Principal: string (nullable = true)
 |-- Descripcion del Proceso: string (nullable = true)
 |-- Tipo de Contrato: string (nullable = true)
 |-- Modalidad de Contratacion: string (nullable = true)
 |-- Justificacion Modalidad de Contratacion: string (nullable = true)
 |-- Fecha de

In [6]:

# Inspección de las variables principales de contratación

columnas_clave = [
    "Nombre Entidad",
    "Proveedor Adjudicado",
    "Valor del Contrato",
    "Fecha de Firma",
    "Departamento",
    "Ciudad",
    "ID Contrato"
]

print("Ejemplos de registros de SECOP II:")

df.select(columnas_clave).show(10, truncate=False)


Ejemplos de registros de SECOP II:
+---------------------------------------------------------------------------------+---------------------------------+------------------+--------------+--------------------------+--------------+------------------+
|Nombre Entidad                                                                   |Proveedor Adjudicado             |Valor del Contrato|Fecha de Firma|Departamento              |Ciudad        |ID Contrato       |
+---------------------------------------------------------------------------------+---------------------------------+------------------+--------------+--------------------------+--------------+------------------+
|ESE HOSPITAL DEPARTAMENTAL UNIVERSITARIO DEL QUINDÍO SAN JUAN DE DIOS            |DIEGO ALEXANDER ALVAREZ PALACIO  |$8.959.088        |11/01/2022    |Quindío                   |Armenia       |CO1.PCCNTR.4168447|
|MUNICIPIO DE SOLEDAD                                                             |ROSA MARIA                    

In [7]:

from pyspark.sql.functions import (
    col, trim, regexp_replace, when, to_date,
    count, sum as spark_sum, countDistinct
)

# Crear una versión de trabajo con las variables necesarias
df_contratos = df.select(
    trim(col("ID Contrato")).alias("id_contrato"),
    trim(col("Nombre Entidad")).alias("entidad"),
    trim(col("Proveedor Adjudicado")).alias("contratista"),
    trim(col("Departamento")).alias("departamento"),
    trim(col("Ciudad")).alias("ciudad"),
    trim(col("Valor del Contrato")).alias("valor_original"),
    trim(col("Fecha de Firma")).alias("fecha_original")
)

# Convertir el valor del contrato a número
df_contratos = df_contratos.withColumn(
    "valor_contrato",
    regexp_replace(
        regexp_replace(col("valor_original"), r"[$.]",""),
        ",", "."
    ).cast("double")
)

# Identificar fechas que necesitan revisión
df_contratos = df_contratos.withColumn(
    "fecha_ddMMyyyy",
    to_date(col("fecha_original"), "dd/MM/yyyy")
).withColumn(
    "fecha_MMddyyyy",
    to_date(col("fecha_original"), "MM/dd/yyyy")
)

# Conservar una fecha válida cuando alguno de los formatos coincida
df_contratos = df_contratos.withColumn(
    "fecha_firma",
    when(col("fecha_ddMMyyyy").isNotNull(), col("fecha_ddMMyyyy"))
    .otherwise(col("fecha_MMddyyyy"))
)

print("Estructura inicial de contratación preparada")
df_contratos.select(
    "id_contrato",
    "entidad",
    "contratista",
    "valor_contrato",
    "fecha_original",
    "fecha_firma",
    "departamento",
    "ciudad"
).show(10, truncate=False)


Estructura inicial de contratación preparada
+------------------+---------------------------------------------------------------------------------+---------------------------------+--------------+--------------+-----------+--------------------------+--------------+
|id_contrato       |entidad                                                                          |contratista                      |valor_contrato|fecha_original|fecha_firma|departamento              |ciudad        |
+------------------+---------------------------------------------------------------------------------+---------------------------------+--------------+--------------+-----------+--------------------------+--------------+
|CO1.PCCNTR.4168447|ESE HOSPITAL DEPARTAMENTAL UNIVERSITARIO DEL QUINDÍO SAN JUAN DE DIOS            |DIEGO ALEXANDER ALVAREZ PALACIO  |8959088.0     |11/01/2022    |2022-01-11 |Quindío                   |Armenia       |
|CO1.PCCNTR.9324146|MUNICIPIO DE SOLEDAD                               

In [8]:
from pyspark.sql.functions import col, count, when, sum as spark_sum

resumen_calidad = df_contratos.agg(
    count("*").alias("total_registros"),

    spark_sum(
        when(col("id_contrato").isNull() | (col("id_contrato") == ""), 1).otherwise(0)
    ).alias("sin_id_contrato"),

    spark_sum(
        when(col("entidad").isNull() | (col("entidad") == ""), 1).otherwise(0)
    ).alias("sin_entidad"),

    spark_sum(
        when(col("contratista").isNull() | (col("contratista") == ""), 1).otherwise(0)
    ).alias("sin_contratista"),

    spark_sum(
        when(col("valor_contrato").isNull(), 1).otherwise(0)
    ).alias("sin_valor"),

    spark_sum(
        when(col("fecha_firma").isNull(), 1).otherwise(0)
    ).alias("sin_fecha")
)

resumen_calidad.show(truncate=False)

+---------------+---------------+-----------+---------------+---------+---------+
|total_registros|sin_id_contrato|sin_entidad|sin_contratista|sin_valor|sin_fecha|
+---------------+---------------+-----------+---------------+---------+---------+
|1777673        |5              |0          |5              |5        |125974   |
+---------------+---------------+-----------+---------------+---------+---------+



In [9]:
from pyspark.sql.functions import count

# Revisar registros sin identificador
print("Registros sin ID de contrato:")
df_contratos.filter(
    col("id_contrato").isNull() | (col("id_contrato") == "")
).show(10, truncate=False)

# Revisar posibles contratos duplicados
print("Cantidad de IDs de contrato duplicados:")

duplicados = (
    df_contratos
    .filter(col("id_contrato").isNotNull() & (col("id_contrato") != ""))
    .groupBy("id_contrato")
    .agg(count("*").alias("apariciones"))
    .filter(col("apariciones") > 1)
)

print("IDs duplicados:", duplicados.count())

duplicados.orderBy(
    col("apariciones").desc()
).show(10, truncate=False)

Registros sin ID de contrato:
+-----------+----------------------------+-----------+------------+------+--------------+--------------+--------------+--------------+--------------+-----------+
|id_contrato|entidad                     |contratista|departamento|ciudad|valor_original|fecha_original|valor_contrato|fecha_ddMMyyyy|fecha_MMddyyyy|fecha_firma|
+-----------+----------------------------+-----------+------------+------+--------------+--------------+--------------+--------------+--------------+-----------+
|NULL       |{                           |NULL       |NULL        |NULL  |NULL          |NULL          |NULL          |NULL          |NULL          |NULL       |
|NULL       |"error" : true              |NULL       |NULL        |NULL  |NULL          |NULL          |NULL          |NULL          |NULL          |NULL       |
|NULL       |"message" : "Internal error"|NULL       |NULL        |NULL  |NULL          |NULL          |NULL          |NULL          |NULL          |NULL       

In [10]:
# Excluir registros que no tienen un ID de contrato válido
df_limpio = df_contratos.filter(
    col("id_contrato").isNotNull() &
    (col("id_contrato") != "")
)

print("Base limpia preparada")
print("Registros originales:", df_contratos.count())
print("Registros después de limpieza:", df_limpio.count())
print("Registros excluidos:", df_contratos.count() - df_limpio.count())

# Revisar una muestra
df_limpio.select(
    "id_contrato",
    "entidad",
    "contratista",
    "valor_contrato",
    "fecha_firma",
    "departamento",
    "ciudad"
).show(5, truncate=False)

Base limpia preparada
Registros originales: 1777673
Registros después de limpieza: 1777668
Registros excluidos: 5
+------------------+---------------------------------------------------------------------+-------------------------------+--------------+-----------+--------------------------+--------------+
|id_contrato       |entidad                                                              |contratista                    |valor_contrato|fecha_firma|departamento              |ciudad        |
+------------------+---------------------------------------------------------------------+-------------------------------+--------------+-----------+--------------------------+--------------+
|CO1.PCCNTR.4168447|ESE HOSPITAL DEPARTAMENTAL UNIVERSITARIO DEL QUINDÍO SAN JUAN DE DIOS|DIEGO ALEXANDER ALVAREZ PALACIO|8959088.0     |2022-01-11 |Quindío                   |Armenia       |
|CO1.PCCNTR.9324146|MUNICIPIO DE SOLEDAD                                                 |ROSA MARIA                  

In [11]:
# Ruta de almacenamiento de la base limpia
ruta_limpia = "s3a://raw/procesado/tercer_corte/contratos_limpios/"

# Guardar los datos en formato Parquet
df_limpio.write \
    .mode("overwrite") \
    .parquet(ruta_limpia)

print("Base limpia guardada correctamente en MinIO")
print("Ruta:", ruta_limpia)

Base limpia guardada correctamente en MinIO
Ruta: s3a://raw/procesado/tercer_corte/contratos_limpios/


In [12]:
# Recuperar la base limpia desde MinIO
df_verificacion = spark.read.parquet(ruta_limpia)

# Comparar cantidades de registros
registros_guardados = df_verificacion.count()
registros_esperados = df_limpio.count()

print("Registros esperados:", registros_esperados)
print("Registros recuperados:", registros_guardados)

if registros_guardados == registros_esperados:
    print("VERIFICACIÓN EXITOSA: los datos se conservaron correctamente.")
else:
    print("ADVERTENCIA: las cantidades de registros no coinciden.")

Registros esperados: 1777668
Registros recuperados: 1777668
VERIFICACIÓN EXITOSA: los datos se conservaron correctamente.


In [13]:
from pyspark.sql.functions import year, month, count, sum as spark_sum

# Crear variables temporales
df_temporal = df_verificacion.withColumn(
    "anio", year(col("fecha_firma"))
).withColumn(
    "mes", month(col("fecha_firma"))
)

# Resumen mensual
resumen_mensual = (
    df_temporal
    .filter(col("fecha_firma").isNotNull())
    .groupBy("anio", "mes")
    .agg(
        count("*").alias("cantidad_contratos"),
        spark_sum("valor_contrato").alias("valor_total")
    )
    .orderBy("anio", "mes")
)

print("Resumen mensual de contratación:")
resumen_mensual.show(15, truncate=False)

# Resumen anual
resumen_anual = (
    df_temporal
    .filter(col("fecha_firma").isNotNull())
    .groupBy("anio")
    .agg(
        count("*").alias("cantidad_contratos"),
        spark_sum("valor_contrato").alias("valor_total")
    )
    .orderBy("anio")
)

print("Resumen anual de contratación:")
resumen_anual.show(truncate=False)

Resumen mensual de contratación:
+----+---+------------------+---------------------+
|anio|mes|cantidad_contratos|valor_total          |
+----+---+------------------+---------------------+
|2015|3  |3                 |7.12680101E8         |
|2015|12 |1                 |1509710.0            |
|2016|1  |38                |2.026184774E9        |
|2016|2  |15                |3.99018414E9         |
|2016|3  |12                |2.44036682E8         |
|2016|4  |30                |1.49439145105E9      |
|2016|5  |30                |1.073712091E9        |
|2016|6  |24                |6.65182581E9         |
|2016|7  |30                |9.1369925596E8       |
|2016|8  |37                |5.5040678718E9       |
|2016|9  |33                |8.9789219219E8       |
|2016|10 |43                |1.85598816259E9      |
|2016|11 |34                |2.912418569E9        |
|2016|12 |81                |1.2141346959710001E10|
|2017|1  |525               |3.167522236467118E13 |
+----+---+------------------+--

In [14]:
print("Los 15 contratos con mayor valor registrado:")

df_verificacion.select(
    "id_contrato",
    "entidad",
    "contratista",
    "valor_original",
    "valor_contrato",
    "fecha_firma"
).orderBy(
    col("valor_contrato").desc()
).show(15, truncate=False)

Los 15 contratos con mayor valor registrado:
+------------------+-------------------------------------------------------------+------------------------------------------------------------------------------------------------------+-----------------------------+----------------------+-----------+
|id_contrato       |entidad                                                      |contratista                                                                                           |valor_original               |valor_contrato        |fecha_firma|
+------------------+-------------------------------------------------------------+------------------------------------------------------------------------------------------------------+-----------------------------+----------------------+-----------+
|CO1.PCCNTR.7326381|INSTITUCIÓN UNIVERSITARIA ITM                                |ANA ALZATE                                                                                            |$734.278.717.259.

In [15]:
from pyspark.sql.functions import (
    percentile_approx, count, when, col,
    length, regexp_replace
)

# Calcular distribución de los valores monetarios
print("Distribución de valores de contratos:")

df_verificacion.select(
    percentile_approx("valor_contrato", 0.50).alias("mediana"),
    percentile_approx("valor_contrato", 0.90).alias("percentil_90"),
    percentile_approx("valor_contrato", 0.95).alias("percentil_95"),
    percentile_approx("valor_contrato", 0.99).alias("percentil_99"),
    percentile_approx("valor_contrato", 0.999).alias("percentil_99_9")
).show(truncate=False)

# Contar valores extremadamente grandes
print("Cantidad de registros por rangos:")

df_verificacion.select(
    count("*").alias("total_registros"),

    count(when(col("valor_contrato") >= 1e12, True)).alias("mayores_a_un_billon"),

    count(when(col("valor_contrato") >= 1e15, True)).alias("mayores_a_mil_billones"),

    count(when(col("valor_contrato").isNull(), True)).alias("valores_nulos")
).show(truncate=False)

Distribución de valores de contratos:
+--------+------------+------------+------------+---------------+
|mediana |percentil_90|percentil_95|percentil_99|percentil_99_9 |
+--------+------------+------------+------------+---------------+
|1.9444E7|8.376E7     |1.87549476E8|2.0E9       |3.9785488519E10|
+--------+------------+------------+------------+---------------+

Cantidad de registros por rangos:
+---------------+-------------------+----------------------+-------------+
|total_registros|mayores_a_un_billon|mayores_a_mil_billones|valores_nulos|
+---------------+-------------------+----------------------+-------------+
|1777668        |1025               |247                   |0            |
+---------------+-------------------+----------------------+-------------+



In [16]:
# Revisar 20 registros con valores superiores a un billón
valores_sospechosos = (
    df_verificacion
    .filter(col("valor_contrato") >= 1e12)
    .select(
        "id_contrato",
        "entidad",
        "valor_original",
        "valor_contrato"
    )
    .orderBy(col("valor_contrato").desc())
)

print("Cantidad de valores sospechosos:", valores_sospechosos.count())

valores_sospechosos.show(20, truncate=False)

Cantidad de valores sospechosos: 1025
+------------------+-----------------------------------------------------------------+-----------------------------+----------------------+
|id_contrato       |entidad                                                          |valor_original               |valor_contrato        |
+------------------+-----------------------------------------------------------------+-----------------------------+----------------------+
|CO1.PCCNTR.7326381|INSTITUCIÓN UNIVERSITARIA ITM                                    |$734.278.717.259.278.397.525 |7.342787172592785E20  |
|CO1.PCCNTR.2632003|Institución Universitaria Pascual Bravo COMPRADOR                |$436.757.482.471.244.431.403 |4.367574824712444E20  |
|CO1.PCCNTR.2038144|ICBF REGIONAL CAUCA                                              |$33.459.047.021.402.144.041  |3.3459047021402145E19 |
|CO1.PCCNTR.8180914|(Secretaría Distrital de Integración Social)                     |$4.371.682.798.701.825.625   |4.3716

In [17]:
from pyspark.sql.functions import when, lit

df_auditoria = df_verificacion.withColumn(
    "estado_valor",
    when(
        col("valor_contrato").isNull(),
        "SIN VALOR"
    ).when(
        col("valor_contrato") >= 1_000_000_000_000,
        "SOSPECHOSO"
    ).otherwise(
        "NORMAL"
    )
)

# Resumen de calidad monetaria
resumen_valores = (
    df_auditoria
    .groupBy("estado_valor")
    .count()
    .orderBy("estado_valor")
)

resumen_valores.show(truncate=False)

+------------+-------+
|estado_valor|count  |
+------------+-------+
|NORMAL      |1776643|
|SOSPECHOSO  |1025   |
+------------+-------+



In [18]:
from pyspark.sql.functions import year, month, concat, lpad

# Preparar registros para la simulación temporal
df_flujo = (
    df_auditoria
    .filter(col("fecha_firma").isNotNull())
    .withColumn("anio", year(col("fecha_firma")))
    .withColumn("mes", month(col("fecha_firma")))
    .withColumn(
        "lote",
        concat(
            col("anio").cast("string"),
            lit("-"),
            lpad(col("mes").cast("string"), 2, "0")
        )
    )
)

print("Registros disponibles para la simulación:", df_flujo.count())
print("Ventanas mensuales identificadas:", df_flujo.select("lote").distinct().count())

print("Primeras ventanas del flujo:")
df_flujo.select("lote").distinct().orderBy("lote").show(12, truncate=False)

Registros disponibles para la simulación: 1651699
Ventanas mensuales identificadas: 134
Primeras ventanas del flujo:
+-------+
|lote   |
+-------+
|2015-03|
|2015-12|
|2016-01|
|2016-02|
|2016-03|
|2016-04|
|2016-05|
|2016-06|
|2016-07|
|2016-08|
|2016-09|
|2016-10|
+-------+
only showing top 12 rows



In [19]:
from pyspark.sql.functions import sum as spark_sum, count, when

resumen_lotes = (
    df_flujo
    .groupBy("lote")
    .agg(
        count("*").alias("contratos_recibidos"),

        spark_sum(
            when(col("estado_valor") == "NORMAL", 1).otherwise(0)
        ).alias("contratos_normales"),

        spark_sum(
            when(col("estado_valor") == "SOSPECHOSO", 1).otherwise(0)
        ).alias("contratos_sospechosos"),

        spark_sum(
            when(
                col("estado_valor") == "NORMAL",
                col("valor_contrato")
            ).otherwise(0)
        ).alias("valor_normal"),

        spark_sum(
            when(
                col("estado_valor") == "SOSPECHOSO",
                col("valor_contrato")
            ).otherwise(0)
        ).alias("valor_en_revision")
    )
    .orderBy("lote")
)

print("Cantidad de lotes mensuales:", resumen_lotes.count())

print("Estadísticas de los primeros 15 lotes:")
resumen_lotes.show(15, truncate=False)

Cantidad de lotes mensuales: 134
Estadísticas de los primeros 15 lotes:
+-------+-------------------+------------------+---------------------+---------------------+-----------------+
|lote   |contratos_recibidos|contratos_normales|contratos_sospechosos|valor_normal         |valor_en_revision|
+-------+-------------------+------------------+---------------------+---------------------+-----------------+
|2015-03|3                  |3                 |0                    |7.12680101E8         |0.0              |
|2015-12|1                  |1                 |0                    |1509710.0            |0.0              |
|2016-01|38                 |38                |0                    |2.026184774E9        |0.0              |
|2016-02|15                 |15                |0                    |3.99018414E9         |0.0              |
|2016-03|12                 |12                |0                    |2.44036682E8         |0.0              |
|2016-04|30                 |30         

In [20]:
# Configurar sobrescritura dinámica de particiones
spark.conf.set(
    "spark.sql.sources.partitionOverwriteMode",
    "dynamic"
)

# Seleccionar el primer lote mensual
lote_actual = "2015-03"

df_lote = df_flujo.filter(
    col("lote") == lote_actual
)

# Ruta persistente del flujo procesado
ruta_flujo = "s3a://raw/procesado/tercer_corte/flujo_mensual/"

# Guardar el lote particionado por mes
df_lote.write \
    .mode("overwrite") \
    .partitionBy("lote") \
    .parquet(ruta_flujo)

print("Lote procesado:", lote_actual)
print("Contratos recibidos:", df_lote.count())
print("Datos guardados en MinIO:", ruta_flujo)

Lote procesado: 2015-03
Contratos recibidos: 3
Datos guardados en MinIO: s3a://raw/procesado/tercer_corte/flujo_mensual/


In [21]:
# Leer el lote que acabamos de guardar
df_lote_guardado = spark.read.parquet(ruta_flujo)

# Filtrar el primer mes
df_lote_verificado = df_lote_guardado.filter(
    col("lote") == lote_actual
)

cantidad_original = df_lote.count()
cantidad_guardada = df_lote_verificado.count()

print("Lote:", lote_actual)
print("Contratos originales:", cantidad_original)
print("Contratos recuperados:", cantidad_guardada)

if cantidad_original == cantidad_guardada:
    print("VERIFICACIÓN EXITOSA: el lote se almacenó correctamente.")
else:
    print("ERROR: las cantidades no coinciden.")

df_lote_verificado.select(
    "id_contrato",
    "entidad",
    "contratista",
    "valor_contrato",
    "lote"
).show(truncate=False)

Lote: 2015-03
Contratos originales: 3
Contratos recuperados: 3
VERIFICACIÓN EXITOSA: el lote se almacenó correctamente.
+---------------+----------------------------------------------+------------------------------------------+--------------+-------+
|id_contrato    |entidad                                       |contratista                               |valor_contrato|lote   |
+---------------+----------------------------------------------+------------------------------------------+--------------+-------+
|CO1.PCCNTR.6602|MINISTERIO DE AMBIENTE Y DESARROLLO SOSTENIBLE|UT MAPFRE - PREVISORA                     |1.45120101E8  |2015-03|
|CO1.PCCNTR.5201|MINISTERIO DE AMBIENTE Y DESARROLLO SOSTENIBLE|CORPORACIÓN ECOVERSA                      |5.2E8         |2015-03|
|CO1.PCCNTR.6604|MINISTERIO DE AMBIENTE Y DESARROLLO SOSTENIBLE|Chubb de Colombia Compañía de Seguros S.A.|4.756E7       |2015-03|
+---------------+----------------------------------------------+------------------------------

In [22]:
# Reprocesar exactamente el mismo lote
df_lote.write \
    .mode("overwrite") \
    .partitionBy("lote") \
    .parquet(ruta_flujo)

# Recuperar nuevamente los datos
df_despues_reproceso = spark.read.parquet(ruta_flujo)

# Contar los contratos del lote
cantidad_final = (
    df_despues_reproceso
    .filter(col("lote") == lote_actual)
    .count()
)

print("Lote reprocesado:", lote_actual)
print("Contratos en el lote original:", cantidad_original)
print("Contratos después del reproceso:", cantidad_final)

if cantidad_final == cantidad_original:
    print("PRUEBA EXITOSA: el reproceso no generó duplicados.")
else:
    print("ADVERTENCIA: la cantidad de registros cambió.")

Lote reprocesado: 2015-03
Contratos en el lote original: 3
Contratos después del reproceso: 3
PRUEBA EXITOSA: el reproceso no generó duplicados.


In [23]:
# Mantener la sobrescritura dinámica de particiones
spark.conf.set(
    "spark.sql.sources.partitionOverwriteMode",
    "dynamic"
)

# Guardar todas las ventanas mensuales en MinIO
df_flujo.write \
    .mode("overwrite") \
    .partitionBy("lote") \
    .parquet(ruta_flujo)

print("Procesamiento de ventanas finalizado.")
print("Cantidad de ventanas esperadas:", resumen_lotes.count())

# Recuperar la información almacenada
df_flujo_guardado = spark.read.parquet(ruta_flujo)

# Verificar cantidad de registros
print("Registros procesados:", df_flujo.count())
print("Registros recuperados:", df_flujo_guardado.count())

# Verificar cantidad de ventanas
print(
    "Ventanas almacenadas:",
    df_flujo_guardado.select("lote").distinct().count()
)

Procesamiento de ventanas finalizado.
Cantidad de ventanas esperadas: 134
Registros procesados: 1651699
Registros recuperados: 1651699
Ventanas almacenadas: 134


In [24]:
# Ruta independiente para probar llegadas tardías
ruta_prueba_tardia = (
    "s3a://raw/procesado/tercer_corte/prueba_llegada_tardia/"
)

# Seleccionar los 3 contratos de marzo de 2015
contratos_marzo = df_flujo.filter(
    col("lote") == "2015-03"
)

# Simular que el primer lote llegó solamente con 2 contratos
contrato_tardio = contratos_marzo.orderBy(
    "id_contrato"
).limit(1)

lote_inicial = contratos_marzo.join(
    contrato_tardio.select("id_contrato"),
    on="id_contrato",
    how="left_anti"
)

# Seleccionar diciembre como otro período histórico
lote_diciembre = df_flujo.filter(
    col("lote") == "2015-12"
)

# Guardar ambos períodos en la ruta de prueba
lote_inicial.unionByName(lote_diciembre).write \
    .mode("overwrite") \
    .partitionBy("lote") \
    .parquet(ruta_prueba_tardia)

print("Escenario inicial preparado")
print("Contratos iniciales de marzo:", lote_inicial.count())
print("Contratos de diciembre:", lote_diciembre.count())
print("Contrato reservado para llegada tardía:")
contrato_tardio.select(
    "id_contrato", "entidad", "lote"
).show(truncate=False)

Escenario inicial preparado
Contratos iniciales de marzo: 2
Contratos de diciembre: 1
Contrato reservado para llegada tardía:
+---------------+----------------------------------------------+-------+
|id_contrato    |entidad                                       |lote   |
+---------------+----------------------------------------------+-------+
|CO1.PCCNTR.5201|MINISTERIO DE AMBIENTE Y DESARROLLO SOSTENIBLE|2015-03|
+---------------+----------------------------------------------+-------+



In [25]:
# Leer los datos que ya están almacenados
df_estado_inicial = spark.read.parquet(ruta_prueba_tardia)

# Incorporar el contrato que llegó tarde
df_actualizado = df_estado_inicial.unionByName(contrato_tardio)

# Evitar duplicados por identificador de contrato
df_actualizado = df_actualizado.dropDuplicates(["id_contrato"])

# Actualizar las particiones afectadas en MinIO
spark.conf.set(
    "spark.sql.sources.partitionOverwriteMode",
    "dynamic"
)

df_actualizado.write \
    .mode("overwrite") \
    .partitionBy("lote") \
    .parquet(ruta_prueba_tardia)

print("Llegada tardía procesada correctamente")

Llegada tardía procesada correctamente


In [26]:
# Recuperar los datos después de la actualización
df_resultado_tardio = spark.read.parquet(ruta_prueba_tardia)

# Contar contratos por mes
verificacion_tardia = (
    df_resultado_tardio
    .groupBy("lote")
    .count()
    .orderBy("lote")
)

print("Cantidad de contratos después de la llegada tardía:")
verificacion_tardia.show(truncate=False)

# Comprobar los resultados esperados
marzo_final = (
    df_resultado_tardio
    .filter(col("lote") == "2015-03")
    .count()
)

diciembre_final = (
    df_resultado_tardio
    .filter(col("lote") == "2015-12")
    .count()
)

print("Contratos en marzo:", marzo_final)
print("Contratos en diciembre:", diciembre_final)

if marzo_final == 3 and diciembre_final == 1:
    print("PRUEBA EXITOSA: llegada tardía incorporada sin alterar otros períodos.")
else:
    print("ADVERTENCIA: revisar los resultados.")

Cantidad de contratos después de la llegada tardía:
+-------+-----+
|lote   |count|
+-------+-----+
|2015-03|3    |
|2015-12|1    |
+-------+-----+

Contratos en marzo: 3
Contratos en diciembre: 1
PRUEBA EXITOSA: llegada tardía incorporada sin alterar otros períodos.


In [27]:
# Simular que vuelve a llegar el mismo contrato
df_reproceso_tardio = (
    df_resultado_tardio
    .unionByName(contrato_tardio)
    .dropDuplicates(["id_contrato"])
)

# Actualizar nuevamente el almacenamiento
df_reproceso_tardio.write \
    .mode("overwrite") \
    .partitionBy("lote") \
    .parquet(ruta_prueba_tardia)

# Recuperar y verificar
df_final_resiliencia = spark.read.parquet(ruta_prueba_tardia)

marzo_reprocesado = (
    df_final_resiliencia
    .filter(col("lote") == "2015-03")
    .count()
)

print("Contratos de marzo después del segundo envío:", marzo_reprocesado)

if marzo_reprocesado == 3:
    print("PRUEBA EXITOSA: no se generaron duplicados.")
else:
    print("ADVERTENCIA: se detectó una inconsistencia.")

Contratos de marzo después del segundo envío: 3
PRUEBA EXITOSA: no se generaron duplicados.


In [28]:
from pyspark.sql.functions import count, sum as spark_sum

resumen_departamentos = (
    df_auditoria
    .filter(col("estado_valor") == "NORMAL")
    .groupBy("departamento")
    .agg(
        count("*").alias("cantidad_contratos"),
        spark_sum("valor_contrato").alias("valor_total")
    )
    .orderBy(col("valor_total").desc())
)

print("Top 15 departamentos por valor de contratación:")

resumen_departamentos.show(15, truncate=False)

Top 15 departamentos por valor de contratación:
+--------------------------+------------------+---------------------+
|departamento              |cantidad_contratos|valor_total          |
+--------------------------+------------------+---------------------+
|Distrito Capital de Bogotá|596850            |1.3916642746257908E14|
|Antioquia                 |168153            |3.485657652990737E13 |
|Valle del Cauca           |177681            |1.6756957745516844E13|
|Cundinamarca              |76254             |1.0679331175533348E13|
|Atlántico                 |60023             |1.0501205164595488E13|
|Bolívar                   |49629             |7.78383028719507E12  |
|Santander                 |80539             |7.55381243349871E12  |
|Boyacá                    |43042             |7.209343455489478E12 |
|No Definido               |35663             |6.486109121333892E12 |
|Tolima                    |47630             |5.76981123671403E12  |
|Magdalena                 |35432         

In [29]:
resumen_entidades = (
    df_auditoria
    .filter(
        (col("estado_valor") == "NORMAL") &
        col("entidad").isNotNull()
    )
    .groupBy("entidad")
    .agg(
        count("*").alias("cantidad_contratos"),
        spark_sum("valor_contrato").alias("valor_total")
    )
    .orderBy(col("valor_total").desc())
)

print("Top 15 entidades por valor de contratación:")

resumen_entidades.show(15, truncate=False)

Top 15 entidades por valor de contratación:
+-----------------------------------------------------------------+------------------+---------------------+
|entidad                                                          |cantidad_contratos|valor_total          |
+-----------------------------------------------------------------+------------------+---------------------+
|INVIAS                                                           |7552              |1.261971525929973E13 |
|DISTRITO ESPECIAL DE CIENCIA TECNOLOGIA E INNOVACION DE MEDELLIN |4224              |9.61707874486136E12  |
|INSTITUTO DE DESARROLLO URBANO                                   |4350              |5.55015976445012E12  |
|DEPARTAMENTO DE ANTIOQUIA//                                      |3161              |5.13451098069895E12  |
|UNP                                                              |4905              |4.2489987139213096E12|
|DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA         |14313            

In [30]:
resumen_contratistas = (
    df_auditoria
    .filter(
        (col("estado_valor") == "NORMAL") &
        col("contratista").isNotNull() &
        (col("contratista") != "")
    )
    .groupBy("contratista")
    .agg(
        count("*").alias("cantidad_contratos"),
        spark_sum("valor_contrato").alias("valor_total")
    )
    .orderBy(col("valor_total").desc())
)

print("Top 15 contratistas por valor de contratación:")

resumen_contratistas.show(15, truncate=False)

Top 15 contratistas por valor de contratación:
+-----------------------------------------------------------------------------------+------------------+---------------------+
|contratista                                                                        |cantidad_contratos|valor_total          |
+-----------------------------------------------------------------------------------+------------------+---------------------+
|FINANCIERA DE DESARROLLO TERRITORIAL S.A.                                          |93                |2.87518475686117E12  |
|Sin Descripcion                                                                    |46969             |2.0079922841771099E12|
|BANCOLOMBIA                                                                        |69                |1.99895478534606E12  |
|AGENCIA LOGISTICA DE LAS FUERZAS MILITARES                                         |71                |1.57637027014427E12  |
|PROMESA DE E.S.P. FUTURA BOGOTÁ LIMPIA S.A.S..                 

In [31]:
from pyspark.sql.functions import year, sum as spark_sum, count

resumen_anual_dashboard = (
    df_auditoria
    .filter(
        (col("estado_valor") == "NORMAL") &
        col("fecha_firma").isNotNull()
    )
    .withColumn("anio", year(col("fecha_firma")))
    .groupBy("anio")
    .agg(
        count("*").alias("cantidad_contratos"),
        spark_sum("valor_contrato").alias("valor_total")
    )
    .orderBy("anio")
)

print("Evolución anual de la contratación estatal:")
resumen_anual_dashboard.show(20, truncate=False)

Evolución anual de la contratación estatal:
+----+------------------+---------------------+
|anio|cantidad_contratos|valor_total          |
+----+------------------+---------------------+
|2015|4                 |7.14189811E8         |
|2016|407               |3.9705747959299995E10|
|2017|6737              |1.2395052548820317E12|
|2018|42568             |9.25170127624453E12  |
|2019|42297             |7.941273929615459E12 |
|2020|105757            |1.7572674780728258E13|
|2021|167131            |3.1902181921109867E13|
|2022|210937            |2.9520163516456215E13|
|2023|250878            |3.624612076080543E13 |
|2024|282904            |3.8163939388400805E13|
|2025|312381            |5.301052284649149E13 |
|2026|229673            |2.6050828054441062E13|
+----+------------------+---------------------+



In [32]:
# Carpeta principal de indicadores
ruta_indicadores = (
    "s3a://raw/procesado/tercer_corte/indicadores/"
)

# Guardar los cuatro indicadores
resumen_departamentos.write.mode("overwrite").parquet(
    ruta_indicadores + "departamentos/"
)

resumen_entidades.write.mode("overwrite").parquet(
    ruta_indicadores + "entidades/"
)

resumen_contratistas.write.mode("overwrite").parquet(
    ruta_indicadores + "contratistas/"
)

resumen_anual_dashboard.write.mode("overwrite").parquet(
    ruta_indicadores + "evolucion_anual/"
)

print("Los cuatro indicadores fueron guardados correctamente.")
print("Ruta:", ruta_indicadores)

Los cuatro indicadores fueron guardados correctamente.
Ruta: s3a://raw/procesado/tercer_corte/indicadores/


In [1]:
import sys
import os

print("Python del notebook:", sys.executable)
print("SPARK_HOME:", os.environ.get("SPARK_HOME"))
print("PYTHONPATH:", os.environ.get("PYTHONPATH"))

Python del notebook: /opt/conda/bin/python
SPARK_HOME: /usr/local/spark
PYTHONPATH: /usr/local/spark/python/lib/py4j-0.10.9.7-src.zip:/usr/local/spark/python:


In [2]:
# Crear carpeta para los archivos del dashboard
import os

carpeta_dashboard = "/home/jovyan/work/dashboard/datos"
os.makedirs(carpeta_dashboard, exist_ok=True)

# Exportar cada indicador desde MinIO
indicadores = {
    "departamentos": "departamentos",
    "entidades": "entidades",
    "contratistas": "contratistas",
    "evolucion_anual": "evolucion_anual"
}

for nombre, carpeta in indicadores.items():

    df_indicador = spark.read.parquet(
        ruta_indicadores + carpeta + "/"
    )

    df_indicador.toPandas().to_csv(
        f"{carpeta_dashboard}/{nombre}.csv",
        index=False
    )

    print(nombre, "exportado:", df_indicador.count(), "registros")

print("Exportación de indicadores finalizada.")

NameError: name 'spark' is not defined

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("PAMD_Tercer_Corte")
    .config(
        "spark.jars",
        "/home/jovyan/work/jars/hadoop-aws-3.3.4.jar,"
        "/home/jovyan/work/jars/aws-java-sdk-bundle-1.11.1026.jar"
    )
    .getOrCreate()
)

hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.access.key", "minioadmin")
hadoop_conf.set("fs.s3a.secret.key", "minioadmin")
hadoop_conf.set("fs.s3a.path.style.access", "true")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")

print("Spark recuperado:", spark.version)

Spark recuperado: 3.5.0


In [4]:
indicadores = {
    "departamentos": "departamentos",
    "entidades": "entidades",
    "contratistas": "contratistas",
    "evolucion_anual": "evolucion_anual"
}

for nombre, carpeta in indicadores.items():
    df_indicador = spark.read.parquet(
        ruta_indicadores + carpeta + "/"
    )
    print(nombre, ":", df_indicador.count(), "registros")

departamentos : 34 registros
entidades : 5279 registros
contratistas : 776479 registros
evolucion_anual : 12 registros


In [5]:
import os

carpeta_dashboard = "/home/jovyan/work/dashboard/datos"
os.makedirs(carpeta_dashboard, exist_ok=True)

# Departamentos
df_departamentos = spark.read.parquet(
    ruta_indicadores + "departamentos/"
)
df_departamentos.toPandas().to_csv(
    f"{carpeta_dashboard}/departamentos.csv", index=False
)

# Entidades
df_entidades = spark.read.parquet(
    ruta_indicadores + "entidades/"
)
df_entidades.toPandas().to_csv(
    f"{carpeta_dashboard}/entidades.csv", index=False
)

# Top 100 contratistas por valor
df_contratistas = spark.read.parquet(
    ruta_indicadores + "contratistas/"
)
df_contratistas.orderBy(
    "valor_total", ascending=False
).limit(100).toPandas().to_csv(
    f"{carpeta_dashboard}/contratistas.csv", index=False
)

# Evolución anual
df_anual = spark.read.parquet(
    ruta_indicadores + "evolucion_anual/"
)
df_anual.toPandas().to_csv(
    f"{carpeta_dashboard}/evolucion_anual.csv", index=False
)

print("¡Exportación finalizada!")
print("Archivos guardados en:", carpeta_dashboard)

¡Exportación finalizada!
Archivos guardados en: /home/jovyan/work/dashboard/datos


In [3]:
flujo = spark.read.parquet(
    "s3a://raw/procesado/tercer_corte/flujo_mensual/"
)

print("ESQUEMA DE LOS DATOS:")
flujo.printSchema()

print("EJEMPLOS DE REGISTROS:")
flujo.show(10, truncate=False)

print("LOTES DISPONIBLES:")
flujo.select("lote").distinct().orderBy("lote").show(20, truncate=False)

ESQUEMA DE LOS DATOS:
root
 |-- id_contrato: string (nullable = true)
 |-- entidad: string (nullable = true)
 |-- contratista: string (nullable = true)
 |-- departamento: string (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- valor_original: string (nullable = true)
 |-- fecha_original: string (nullable = true)
 |-- valor_contrato: double (nullable = true)
 |-- fecha_ddMMyyyy: date (nullable = true)
 |-- fecha_MMddyyyy: date (nullable = true)
 |-- fecha_firma: date (nullable = true)
 |-- estado_valor: string (nullable = true)
 |-- anio: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- lote: string (nullable = true)

EJEMPLOS DE REGISTROS:
+------------------+---------------------------------------------------------------+--------------------------------+--------------------------+-----------+--------------+--------------+--------------+--------------+--------------+-----------+------------+----+---+-------+
|id_contrato       |entidad                     

In [4]:
from pyspark.sql.functions import (
    col, count, when, sum as spark_sum, concat, lit
)
import os

# Resumir las ventanas mensuales que ya están guardadas en MinIO
mensual = (
    flujo.groupBy("lote")
    .agg(
        count("*").alias("contratos_recibidos"),
        spark_sum(
            when(col("estado_valor") == "NORMAL", 1).otherwise(0)
        ).alias("contratos_normales"),
        spark_sum(
            when(col("estado_valor") == "SOSPECHOSO", 1).otherwise(0)
        ).alias("contratos_sospechosos"),
        spark_sum(
            when(col("estado_valor") == "NORMAL", col("valor_contrato"))
            .otherwise(0)
        ).alias("valor_normal"),
        spark_sum(
            when(col("estado_valor") == "SOSPECHOSO", col("valor_contrato"))
            .otherwise(0)
        ).alias("valor_en_revision")
    )
    .withColumn("mes", concat(col("lote"), lit("-01")))
    .select(
        "mes",
        "contratos_recibidos",
        "contratos_normales",
        "contratos_sospechosos",
        "valor_normal",
        "valor_en_revision"
    )
    .orderBy("mes")
)

# Exportar el resumen: son solo 134 filas
carpeta = "/home/jovyan/work/dashboard/datos"
os.makedirs(carpeta, exist_ok=True)

mensual.toPandas().to_csv(
    carpeta + "/flujo_mensual.csv",
    index=False
)

print("¡EXPORTACIÓN MENSUAL FINALIZADA!")
print("Ventanas exportadas:", mensual.count())
print("Primeros meses:")
mensual.show(10, truncate=False)

¡EXPORTACIÓN MENSUAL FINALIZADA!
Ventanas exportadas: 134
Primeros meses:
+----------+-------------------+------------------+---------------------+---------------+-----------------+
|mes       |contratos_recibidos|contratos_normales|contratos_sospechosos|valor_normal   |valor_en_revision|
+----------+-------------------+------------------+---------------------+---------------+-----------------+
|2015-03-01|3                  |3                 |0                    |7.12680101E8   |0.0              |
|2015-12-01|1                  |1                 |0                    |1509710.0      |0.0              |
|2016-01-01|38                 |38                |0                    |2.026184774E9  |0.0              |
|2016-02-01|15                 |15                |0                    |3.99018414E9   |0.0              |
|2016-03-01|12                 |12                |0                    |2.44036682E8   |0.0              |
|2016-04-01|30                 |30                |0          